# Distillation through augmented noise — AlexNet → AlexNet-Half (CIFAR-10)

Data-free knowledge distillation: synthetic noise images (smooth gradient / Perlin /
uniform / Gabor / checkerboard) are pushed through a 17-op augmentation stack, the frozen
teacher labels the augmented view on the fly, and the student is trained against those
labels with a temperature-scaled KL objective under a batch-size ramp and phase-aligned
cosine LR schedule. No CIFAR-10 training image ever reaches the student — the real data is
used only to train the teacher and to score both networks.

## Architectures

From the ZSKD supplementary,
[Table 2](https://proceedings.mlr.press/v97/nayak19a/nayak19a-supp.pdf)
(Nayak et al., *Zero-Shot Knowledge Distillation in Deep Networks*, ICML 2019),
cross-checked against the authors' released code
([vcl-iisc/ZSKD](https://github.com/vcl-iisc/ZSKD), `model_alex_full.py` /
`model_alex_half.py`).

> *AlexNet-Half is derived from AlexNet by taking half of the convolutional filters
> and half of the neurons in the fully connected layers, except in the
> classification layer.*

| Layer | Kernel / stride | Output (32×32 in) | AlexNet (teacher) | AlexNet-Half (student) |
|---|---|---|---|---|
| conv1 + ReLU + LRN | 5×5, s1, `SAME` | 32×32 | 48 | 24 |
| maxpool1 + BN | 3×3, s2, `VALID` | 15×15 | — | — |
| conv2 + ReLU + LRN | 5×5, s1, `SAME` | 15×15 | 128 | 64 |
| maxpool2 + BN | 3×3, s2, `VALID` | 7×7 | — | — |
| conv3 + ReLU + BN | 3×3, s1, `SAME` | 7×7 | 192 | 96 |
| conv4 + ReLU + BN | 3×3, s1, `SAME` | 7×7 | 192 | 96 |
| conv5 + ReLU | 3×3, s1, `SAME` | 7×7 | 128 | 64 |
| maxpool5 + BN | 3×3, s2, `VALID` | 3×3 | — | — |
| fc1 + ReLU + dropout(0.5) + BN | — | — | 512 | 256 |
| fc2 + ReLU + dropout(0.5) + BN | — | — | 256 | 128 |
| fc3 (classifier) | — | — | 10 | 10 |

Built as specified, the teacher has **1,659,178** parameters and the student **417,434**.
The teacher matches the 1.65 × 10⁶ the paper reports. The student does not: the paper
reports 7.23 × 10⁵, but halving every conv filter and every hidden FC width — what the
paper's own text says, and what the released `model_alex_half.py` does — gives 4.17 × 10⁵.
This follows the text and the code; the reported student figure appears to be an error.

Two notes on porting the reference TensorFlow code to PyTorch:

* **LRN.** TF's `local_response_normalization(depth_radius=2, alpha=1e-4, beta=0.75,
  bias=1.0)` sums over a 5-channel window and does *not* divide `alpha` by the window
  size, while `torch.nn.LocalResponseNorm` does. The port passes `alpha * 5` so the two
  are numerically identical.
* **Init.** The reference initialises every weight from `N(0, 0.01)`. That is available as
  `paper_init=True`, but the default here is Kaiming, which trains much more reliably
  without TF-era LR tuning.

## 1. Setup

In [ ]:
import math
import os
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True          # fixed 32x32 input size -> free speedup

# Worker/precision settings picked from the hardware actually available. None of
# this touches the training math.
_cpu_count = os.cpu_count() or 2
NUM_WORKERS = max(0, min(16, _cpu_count - 1))
PREFETCH_FACTOR = 2 if NUM_WORKERS <= 2 else (4 if NUM_WORKERS <= 8 else 6)
BF16_OK = device.type == "cuda" and torch.cuda.is_bf16_supported()
torch.set_num_threads(max(1, min(_cpu_count, 32)))
if device.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print(f"Device: {device}")
print(f"NUM_WORKERS={NUM_WORKERS} PREFETCH_FACTOR={PREFETCH_FACTOR} BF16_OK={BF16_OK}")

## 2. AlexNet / AlexNet-Half

PyTorch port of the supplementary Table 2 architectures — see the header cell for the
layer table and the two porting notes.

In [ ]:
class AlexNetCIFAR(nn.Module):
    """AlexNet for 32x32 CIFAR inputs (ZSKD supplementary Table 2).

    Torchvision's AlexNet is an ImageNet 224x224 design that downsamples 32x32
    inputs to nothing, so the paper's CIFAR variant is used instead. Block order
    is the reference implementation's:
        conv -> ReLU -> LRN -> [maxpool] -> BN
        fc   -> ReLU -> dropout -> BN

    widths = (c1, c2, c3, c4, c5, f1, f2)
    """

    def __init__(self, widths, num_classes=10, dropout=0.5, paper_init=False):
        super().__init__()
        c1, c2, c3, c4, c5, f1, f2 = widths

        # TF's LRN sums over a 5-channel window without dividing alpha by it;
        # torch divides by `size`. alpha*5 makes the two identical.
        def lrn():
            return nn.LocalResponseNorm(size=5, alpha=1e-4 * 5, beta=0.75, k=1.0)

        # 'SAME' padding at stride 1: k=5 -> pad 2, k=3 -> pad 1.
        # Pooling is 3x3 / stride 2 'VALID': 32 -> 15 -> 7 -> 3.
        self.conv1, self.lrn1 = nn.Conv2d(3, c1, 5, 1, 2), lrn()
        self.pool1, self.bn1 = nn.MaxPool2d(3, 2), nn.BatchNorm2d(c1)

        self.conv2, self.lrn2 = nn.Conv2d(c1, c2, 5, 1, 2), lrn()
        self.pool2, self.bn2 = nn.MaxPool2d(3, 2), nn.BatchNorm2d(c2)

        self.conv3, self.bn3 = nn.Conv2d(c2, c3, 3, 1, 1), nn.BatchNorm2d(c3)
        self.conv4, self.bn4 = nn.Conv2d(c3, c4, 3, 1, 1), nn.BatchNorm2d(c4)

        self.conv5, self.pool5 = nn.Conv2d(c4, c5, 3, 1, 1), nn.MaxPool2d(3, 2)
        self.bn5 = nn.BatchNorm2d(c5)

        self.fc1, self.drop1 = nn.Linear(3 * 3 * c5, f1), nn.Dropout(dropout)
        self.bn6 = nn.BatchNorm1d(f1)
        self.fc2, self.drop2 = nn.Linear(f1, f2), nn.Dropout(dropout)
        self.bn7 = nn.BatchNorm1d(f2)
        self.fc3 = nn.Linear(f2, num_classes)

        self._init_weights(paper_init)

    def _init_weights(self, paper_init):
        if paper_init:
            # The reference TF init: N(0, 0.01) weights, bias 1.0 on conv2/conv4/
            # conv5 (the original AlexNet convention), 0 elsewhere.
            ones = {self.conv2, self.conv4, self.conv5}
            for m in self.modules():
                if isinstance(m, (nn.Conv2d, nn.Linear)):
                    nn.init.normal_(m.weight, std=0.01)
                    nn.init.constant_(m.bias, 1.0 if m in ones else 0.0)
            return
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.01)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.bn1(self.pool1(self.lrn1(F.relu(self.conv1(x)))))
        x = self.bn2(self.pool2(self.lrn2(F.relu(self.conv2(x)))))
        x = self.bn3(F.relu(self.conv3(x)))
        x = self.bn4(F.relu(self.conv4(x)))
        x = self.bn5(self.pool5(F.relu(self.conv5(x))))
        x = torch.flatten(x, 1)
        x = self.bn6(self.drop1(F.relu(self.fc1(x))))
        x = self.bn7(self.drop2(F.relu(self.fc2(x))))
        return self.fc3(x)


def AlexNet(num_classes=10, **kw):
    """Teacher: the 'AlexNet' row of supplementary Table 2."""
    return AlexNetCIFAR((48, 128, 192, 192, 128, 512, 256), num_classes, **kw)


def AlexNetHalf(num_classes=10, **kw):
    """Student: the 'AlexNet-Half' row -- half the conv filters and half the FC
    neurons, classification layer untouched."""
    return AlexNetCIFAR((24, 64, 96, 96, 64, 256, 128), num_classes, **kw)


for _name, _net in [("AlexNet (teacher)", AlexNet()), ("AlexNet-Half (student)", AlexNetHalf())]:
    print(f"{_name:24s} {sum(p.numel() for p in _net.parameters()):>9,d} params")

## 3. CIFAR-10

`DATA_ROOT` must be the directory that *contains* `cifar-10-batches-py`, not that folder
itself — torchvision resolves the data as `<root>/cifar-10-batches-py/...`.

In [ ]:
NUM_CLASSES = 10
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

DATA_ROOT = '/home/vu-lab03-pc43/Downloads/data'
DOWNLOAD = False

if not DOWNLOAD and not os.path.isdir(os.path.join(DATA_ROOT, 'cifar-10-batches-py')):
    raise FileNotFoundError(
        f"{DATA_ROOT} does not contain 'cifar-10-batches-py'. DATA_ROOT must be the "
        f"PARENT of that folder, not the folder itself. Set DOWNLOAD = True to fetch it."
    )

transform_train = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
])
transform_test = T.Compose([T.ToTensor()])

train_set = torchvision.datasets.CIFAR10(DATA_ROOT, train=True, download=DOWNLOAD, transform=transform_train)
test_set = torchvision.datasets.CIFAR10(DATA_ROOT, train=False, download=DOWNLOAD, transform=transform_test)
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False)
print(f"CIFAR-10: {len(train_set)} train / {len(test_set)} test")


@torch.no_grad()
def evaluate(model, device, loader):
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        correct += (torch.argmax(model(x), dim=1) == y).sum().item()
        total += y.size(0)
    return 100.0 * correct / total


class normalization(nn.Module):
    """Wraps a backbone so it takes [0, 1] images and normalizes internally --
    the noise pipeline works in [0, 1] throughout."""

    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, x):
        mean = torch.tensor(CIFAR_MEAN, device=x.device, dtype=x.dtype).view(1, -1, 1, 1)
        std = torch.tensor(CIFAR_STD, device=x.device, dtype=x.dtype).view(1, -1, 1, 1)
        return self.backbone((x - mean) / std)

## 4. Teacher

Trains a teacher if `TEACHER_CKPT` is missing, loads it otherwise.
`load_teacher_checkpoint` absorbs the usual checkpoint-shape differences: a container dict
(`{'model_state': ..., 'val_acc': ..., 'epoch': ...}`) instead of a bare `state_dict`, a
`module.` prefix from `DataParallel`, and the `backbone.` prefix that `normalization(...)`
adds but a checkpoint saved from the bare backbone lacks.

In [ ]:
_CONTAINER_KEYS = ("model_state", "state_dict", "model", "net", "weights", "teacher")
_META_KEYS = ("val_acc", "acc", "best_acc", "epoch", "epochs")


def _strip(prefix):
    return lambda k: k[len(prefix):] if k.startswith(prefix) else k


_KEY_FIXES = [
    ("as-is", lambda k: k),
    ("add 'backbone.'", lambda k: f"backbone.{k}"),
    ("strip 'backbone.'", _strip("backbone.")),
]


def _candidate_state_dicts(obj):
    """Yield every plausible state_dict inside a checkpoint object."""
    if not isinstance(obj, dict):
        return
    if any(torch.is_tensor(v) for v in obj.values()):
        yield obj
    for k in _CONTAINER_KEYS:
        v = obj.get(k)
        if isinstance(v, dict) and any(torch.is_tensor(t) for t in v.values()):
            yield v


def load_teacher_checkpoint(model, path, device):
    """Load `path` into `model`, reconciling container dicts and key prefixes.
    Raises with the keys it saw when nothing matches, rather than a wall of
    missing-key output."""
    try:
        obj = torch.load(path, map_location=device, weights_only=False)
    except TypeError:                      # torch too old for weights_only
        obj = torch.load(path, map_location=device)

    meta = {k: obj[k] for k in _META_KEYS if isinstance(obj, dict) and k in obj}
    want = model.state_dict()
    best = (-1, None, None)

    for sd in _candidate_state_dicts(obj):
        sd = {_strip("module.")(k): v for k, v in sd.items()}
        for name, fix in _KEY_FIXES:
            cand = {fix(k): v for k, v in sd.items()}
            overlap = len(set(want) & set(cand))
            if overlap > best[0]:
                best = (overlap, name, cand)
            if set(cand) == set(want):
                bad = {k: (tuple(v.shape), tuple(want[k].shape))
                       for k, v in cand.items() if v.shape != want[k].shape}
                if bad:
                    k, (got, exp) = next(iter(bad.items()))
                    raise RuntimeError(
                        f"{path} matches the architecture but not its shapes "
                        f"({len(bad)} tensor(s) differ, e.g. {k}: checkpoint {got} vs "
                        f"model {exp}). A classifier mismatch means the checkpoint was "
                        f"trained on a different dataset."
                    )
                model.load_state_dict(cand)
                print(f"  loaded {path} (keys matched {name})"
                      + (f"\n  checkpoint metadata: {meta}" if meta else ""))
                return model

    overlap, name, cand = best
    raise RuntimeError(
        f"Could not match {path} to this model.\n"
        f"  best attempt ({name}) matched {overlap}/{len(want)} keys\n"
        f"  checkpoint keys (first 5): {sorted(cand)[:5] if cand else 'none found'}\n"
        f"  model keys (first 5):      {sorted(want)[:5]}\n"
        f"  Unrelated names mean the checkpoint is a different architecture."
    )


def train_teacher(teacher, train_loader, test_loader, device, epochs=100, lr=0.01,
                  ckpt_path=None):
    """Standard supervised training, saving the best model on the fly."""
    teacher.to(device)
    opt = torch.optim.SGD(teacher.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    ce = nn.CrossEntropyLoss()
    os.makedirs(os.path.dirname(os.path.abspath(ckpt_path)), exist_ok=True)
    best_acc = 0.0

    for epoch in range(epochs):
        teacher.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            ce(teacher(x), y).backward()
            opt.step()
        sched.step()

        acc = evaluate(teacher, device, test_loader)
        if (epoch + 1) % 10 == 0:
            print(f"[teacher] epoch {epoch + 1}/{epochs}  acc={acc:.2f}%  best={best_acc:.2f}%")
        if acc > best_acc:
            best_acc = acc
            torch.save({"model_state": teacher.state_dict(),
                        "val_acc": acc, "epoch": epoch + 1}, ckpt_path)
            print(f"[teacher] epoch {epoch + 1}/{epochs}  new best {acc:.2f}% -> {ckpt_path}")

    print(f"[teacher] done. best_acc={best_acc:.2f}%")
    return teacher, best_acc

In [ ]:
TEACHER_CKPT = '/home/vu-lab03-pc43/Downloads/ALEXNET_test/TUHIN_ZOPGA_CORRECTED/ALEXNET_FULL_teacher_best.pt'

teacher = normalization(AlexNet(NUM_CLASSES)).to(device)

if os.path.exists(TEACHER_CKPT):
    load_teacher_checkpoint(teacher, TEACHER_CKPT, device)
    teacher.to(device)
else:
    teacher, _ = train_teacher(teacher, train_loader, test_loader, device,
                               epochs=100, lr=0.01, ckpt_path=TEACHER_CKPT)

teacher.eval()
for p in teacher.parameters():
    p.requires_grad_(False)

print(f"teacher accuracy: {evaluate(teacher, device, test_loader):.2f}%")

## 5. Noise

The student never sees a CIFAR image. Each synthetic image is one of five noise families,
colourised between two random colours, built on CPU so DataLoader workers can use them.

In [ ]:
def noise_smooth_gradient(n, size=32):
    """Smooth linear gradient at a random angle between two random colours."""
    yy, xx = torch.meshgrid(torch.linspace(0, 1, size), torch.linspace(0, 1, size), indexing="ij")
    imgs = torch.zeros(n, 3, size, size)
    for i in range(n):
        angle = random.uniform(0, 2 * math.pi)
        grad = xx * math.cos(angle) + yy * math.sin(angle)
        imgs[i] = _colourise(grad)
    return imgs.clamp(0.0, 1.0)


def noise_perlin(n, size=32):
    """Perlin noise field per image, at a coarse/medium/fine cell grid."""
    imgs = torch.zeros(n, 3, size, size)
    for i in range(n):
        imgs[i] = _colourise(_perlin_grid(size, random.choice([2, 4, 8])))
    return imgs.clamp(0.0, 1.0)


def noise_uniform(n, size=32):
    """Plain i.i.d. uniform noise -- no spatial smoothness, unlike the others."""
    return torch.rand(n, 3, size, size).clamp(0.0, 1.0)


def noise_gabor(n, size=32):
    """Gaussian-windowed sinusoidal grating, random orientation/frequency/phase."""
    imgs = torch.zeros(n, 3, size, size)
    yy, xx = torch.meshgrid(torch.linspace(-1, 1, size), torch.linspace(-1, 1, size), indexing="ij")
    for i in range(n):
        theta = random.uniform(0, math.pi)
        freq = random.uniform(2.0, 8.0)
        phase = random.uniform(0, 2 * math.pi)
        sigma = random.uniform(0.3, 0.8)
        x_theta = xx * math.cos(theta) + yy * math.sin(theta)
        y_theta = -xx * math.sin(theta) + yy * math.cos(theta)
        gaussian = torch.exp(-(x_theta ** 2 + y_theta ** 2) / (2 * sigma ** 2))
        imgs[i] = _colourise(gaussian * torch.cos(2 * math.pi * freq * x_theta + phase))
    return imgs.clamp(0.0, 1.0)


def noise_checkerboard(n, size=32):
    """Checkerboard with random cell size and random phase offset per image."""
    imgs = torch.zeros(n, 3, size, size)
    yy, xx = torch.meshgrid(torch.arange(size), torch.arange(size), indexing="ij")
    for i in range(n):
        block = random.choice([2, 4, 8, 16])
        ox, oy = random.randint(0, block - 1), random.randint(0, block - 1)
        imgs[i] = _colourise((((xx + ox) // block) + ((yy + oy) // block)) % 2, norm=False)
    return imgs.clamp(0.0, 1.0)


def _colourise(grad, norm=True):
    """Map a scalar field to a 3-channel image interpolating two random colours."""
    grad = grad.float()
    if norm:
        grad = (grad - grad.min()) / (grad.max() - grad.min() + 1e-8)
    color_a, color_b = torch.rand(3, 1, 1), torch.rand(3, 1, 1)
    return grad.unsqueeze(0) * color_a + (1 - grad.unsqueeze(0)) * color_b


def _perlin_grid(size, res):
    """Single 2D Perlin field, values roughly in [-1, 1]. `size` must divide by `res`."""
    assert size % res == 0, "size must be divisible by res"
    d = size // res
    lin = torch.arange(0, res, 1.0 / d)
    gy, gx = torch.meshgrid(lin, lin, indexing="ij")
    grid = torch.stack((gy % 1, gx % 1), dim=-1)

    angles = 2 * math.pi * torch.rand(res + 1, res + 1)
    grads = torch.stack((torch.cos(angles), torch.sin(angles)), dim=-1)
    tile = lambda g: g.repeat_interleave(d, 0).repeat_interleave(d, 1)

    g00, g10 = tile(grads[:-1, :-1]), tile(grads[1:, :-1])
    g01, g11 = tile(grads[:-1, 1:]), tile(grads[1:, 1:])
    n00 = (torch.stack((grid[..., 0],     grid[..., 1]),     -1) * g00).sum(-1)
    n10 = (torch.stack((grid[..., 0] - 1, grid[..., 1]),     -1) * g10).sum(-1)
    n01 = (torch.stack((grid[..., 0],     grid[..., 1] - 1), -1) * g01).sum(-1)
    n11 = (torch.stack((grid[..., 0] - 1, grid[..., 1] - 1), -1) * g11).sum(-1)

    t = 6 * grid ** 5 - 15 * grid ** 4 + 10 * grid ** 3          # fade curve
    return torch.lerp(torch.lerp(n00, n10, t[..., 0]),
                      torch.lerp(n01, n11, t[..., 0]), t[..., 1])


NOISE_INITIALIZERS = {
    "noise_smooth": noise_smooth_gradient,
    "noise_perlin": noise_perlin,
    "noise_uniform": noise_uniform,
    "noise_gabor": noise_gabor,
    "noise_checkerboard": noise_checkerboard,
}


def noise_data(p, per_call=100):
    """p * per_call synthetic images, noise family drawn per call."""
    fns = list(NOISE_INITIALIZERS.values())
    return torch.cat([random.choice(fns)(per_call) for _ in range(p)], dim=0)

## 6. Augmentation

Two independent switches, applied in order: a geometric base (`RandomCrop(32, pad=4,
reflect)` + `RandomHorizontalFlip`), then `n_random_ops` sampled without replacement from
the 17-op pool. Both happen inside `Dataset.__getitem__`, so the teacher is queried on the
augmented view — crop and flip are inside the query path.

In [ ]:
base_geo_transform = T.Compose([
    T.RandomCrop(32, padding=4, padding_mode='reflect'),
    T.RandomHorizontalFlip(),
])

_perspective_tf = T.RandomPerspective(distortion_scale=0.35, p=1.0)
_zoom_crop_tf = T.RandomResizedCrop(32, scale=(0.65, 1.0), ratio=(0.85, 1.15))
_color_jitter_tf = T.ColorJitter(brightness=0.45, contrast=0.45, saturation=0.45, hue=0.12)
_cutout_tf = T.RandomErasing(p=1.0, scale=(0.02, 0.25), ratio=(0.3, 3.3), value=0.0)


def op_rotate(img):
    return T.functional.rotate(img, random.uniform(-20, 20))


def op_affine(img):
    return T.functional.affine(
        img,
        angle=random.uniform(-12, 12),
        translate=(random.randint(-2, 2), random.randint(-2, 2)),
        scale=random.uniform(0.82, 1.18),
        shear=random.uniform(-12, 12),
    )


def op_gaussian_blur(img):
    return T.functional.gaussian_blur(img, kernel_size=random.choice([3, 5]),
                                      sigma=random.uniform(0.1, 2.2))


def op_equalize(img):
    return T.functional.equalize((img.clamp(0.0, 1.0) * 255).to(torch.uint8)).float() / 255.0


def op_posterize(img):
    img_u8 = (img.clamp(0.0, 1.0) * 255).to(torch.uint8)
    return T.functional.posterize(img_u8, random.choice([3, 4, 5, 6])).float() / 255.0


def op_gaussian_noise(img):
    return (img + torch.randn_like(img) * random.uniform(0.01, 0.08)).clamp(0.0, 1.0)


def op_salt_pepper(img):
    prob = random.uniform(0.01, 0.06)
    mask = torch.rand(1, img.shape[1], img.shape[2], device=img.device)
    out = img.clone()
    out[(mask < prob / 2).expand_as(img)] = 1.0
    out[(mask > 1 - prob / 2).expand_as(img)] = 0.0
    return out


def op_random_color_erase(img):
    out = img.clone()
    eh, ew = random.randint(4, 12), random.randint(4, 12)
    y0 = random.randint(0, img.shape[1] - eh)
    x0 = random.randint(0, img.shape[2] - ew)
    out[:, y0:y0 + eh, x0:x0 + ew] = torch.rand(3, 1, 1, device=img.device)
    return out


AUG_OPS = {
    "rotate": op_rotate,
    "affine": op_affine,
    "perspective": _perspective_tf,
    "zoom_crop": _zoom_crop_tf,
    "color_jitter": _color_jitter_tf,
    "grayscale": lambda img: T.functional.rgb_to_grayscale(img, num_output_channels=3),
    "gaussian_blur": op_gaussian_blur,
    "sharpness": lambda img: T.functional.adjust_sharpness(img, random.uniform(0.0, 3.5)),
    "autocontrast": lambda img: T.functional.autocontrast(img.clamp(0.0, 1.0)),
    "equalize": op_equalize,
    "posterize": op_posterize,
    "solarize": lambda img: T.functional.solarize(img.clamp(0.0, 1.0), random.uniform(0.3, 0.9)),
    "invert": lambda img: T.functional.invert(img.clamp(0.0, 1.0)),
    "gaussian_noise": op_gaussian_noise,
    "salt_pepper": op_salt_pepper,
    "cutout": lambda img: _cutout_tf(img.unsqueeze(0)).squeeze(0),
    "random_color_erase": op_random_color_erase,
}
AUG_OP_NAMES = list(AUG_OPS)
assert len(AUG_OP_NAMES) == 17, f"expected a 17-op pool, found {len(AUG_OP_NAMES)}"


def diverse_augment(img, use_geo=True, n_random_ops=4):
    img = img.clamp(0.0, 1.0)
    if use_geo:
        img = base_geo_transform(img)
    for name in random.sample(AUG_OP_NAMES, k=min(int(n_random_ops), len(AUG_OP_NAMES))):
        img = AUG_OPS[name](img).clamp(0.0, 1.0)
    return img


class SyntheticDataset(Dataset):
    """Yields the augmented noise image; the teacher labels it on the fly."""

    def __init__(self, imgs, use_geo=True, n_random_ops=4):
        self.imgs = imgs
        self.use_geo = use_geo
        self.n_random_ops = n_random_ops

    def __len__(self):
        return self.imgs.shape[0]

    def __getitem__(self, idx):
        return diverse_augment(self.imgs[idx], self.use_geo, self.n_random_ops)

## 7. Distillation

Batch size ramps 16 → 2048 across training; the LR schedule is a cosine decay that resets
at each ramp step. One DataLoader is cached per batch size so the worker pool is not
respawned every epoch.

In [ ]:
def klpga(x, student, t_logits, T):
    """Temperature-scaled KL between student and (frozen) teacher, scaled by T^2."""
    log_prob = F.log_softmax(student(x) / T, dim=-1)
    t_prob = F.softmax(t_logits.detach() / T, dim=-1)
    return F.kl_div(log_prob, t_prob, reduction="batchmean") * T ** 2


_BATCH_RAMP = [(25, 16), (50, 64), (75, 128), (100, 256), (125, 512), (150, 1024)]
_loader_cache = {}


def batch_size_for(epoch):
    for limit, bs in _BATCH_RAMP:
        if epoch < limit:
            return bs
    return 2048


def data_to_loader(data, epoch):
    batch_size = batch_size_for(epoch)
    if batch_size not in _loader_cache:
        kwargs = dict(batch_size=batch_size, shuffle=True, num_workers=NUM_WORKERS,
                      pin_memory=True,
                      drop_last=True)   # constant batch shape -> cudnn.benchmark stays hot
        if NUM_WORKERS > 0:
            kwargs["persistent_workers"] = True
            kwargs["prefetch_factor"] = PREFETCH_FACTOR
        _loader_cache[batch_size] = DataLoader(data, **kwargs)
    return _loader_cache[batch_size]


def batch_aligned_lr(epoch):
    """Cosine decay within each phase, resetting at every batch-size change so the
    schedule lines up with the ramp. Returns a LambdaLR multiplier on lr=0.01."""
    start = 0.001 if 25 <= epoch < 50 else 0.01
    end = 0.00001
    phase_start = min(epoch // 25, 6) * 25
    phase_len = 50 if epoch >= 150 else 25

    t = epoch - phase_start
    cos_factor = 0.5 * (1 + math.cos(math.pi * t / max(phase_len - 1, 1)))
    return (end + (start - end) * cos_factor) / 0.01


def _fused_sgd_kwargs(device):
    # Fused CUDA kernel for the optimizer step; same math, same update rule.
    import inspect
    if device.type == "cuda" and "fused" in inspect.signature(torch.optim.SGD.__init__).parameters:
        return {"fused": True}
    return {}


def _prepare_for_fast(model, device):
    # channels_last is a pure memory-layout change (NHWC vs NCHW); conv math is
    # identical but cudnn kernels for it run faster on modern GPUs.
    return model.to(memory_format=torch.channels_last) if device.type == "cuda" else model


def _fast_input(x, device):
    return x.contiguous(memory_format=torch.channels_last) if device.type == "cuda" else x


def _train_autocast(device):
    # bf16 where supported (no GradScaler needed -- bf16 has fp32's exponent range
    # so it doesn't underflow like fp16); fp16+scaler on older GPUs; fp32 on CPU.
    if device.type == "cuda" and BF16_OK:
        return torch.autocast(device_type="cuda", dtype=torch.bfloat16), False
    if device.type == "cuda":
        return torch.amp.autocast("cuda"), True
    from contextlib import nullcontext
    return nullcontext(), False


def train_student(teacher, student, dataset, test_loader, device, T,
                  student_epochs=100, lr=0.001):
    student = _prepare_for_fast(student, device)
    teacher = _prepare_for_fast(teacher, device)
    teacher.eval()      # AlexNet has dropout as well as BN -- keep the labeller deterministic

    opt = torch.optim.SGD(student.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4,
                          **_fused_sgd_kwargs(device))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda=batch_aligned_lr)
    autocast_ctx, needs_scaler = _train_autocast(device)
    scaler = torch.amp.GradScaler("cuda", enabled=needs_scaler)

    for epoch in range(student_epochs):
        student.train()
        loader = data_to_loader(dataset, epoch)
        epoch_loss = 0.0
        for x in loader:
            x = _fast_input(x.to(device, non_blocking=True), device)
            with torch.no_grad(), autocast_ctx:   # teacher already frozen; no_grad also
                z = teacher(x)                    # stops autograd tracking the input side
            opt.zero_grad(set_to_none=True)
            with autocast_ctx:
                loss = klpga(x, student, z, T)
            if needs_scaler:
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            else:
                loss.backward()
                opt.step()
            epoch_loss += loss.item()
        sched.step()

        acc = evaluate(student, device, test_loader)
        print(f"epoch {epoch + 1}/{student_epochs}  acc={acc:.2f}%  "
              f"loss={epoch_loss / len(loader):.4f}  bs={batch_size_for(epoch)}  "
              f"lr={sched.get_last_lr()[0]:.2e}")

    return student

## 8. Run

In [ ]:
student = normalization(AlexNetHalf(NUM_CLASSES)).to(device)

data = noise_data(1500)
print(f"synthetic set: {tuple(data.shape)}")

dataset = SyntheticDataset(data, use_geo=True, n_random_ops=8)

In [ ]:
train_student(teacher, student, dataset, test_loader, device, T=20,
              student_epochs=200, lr=0.01)